# 02 — Instruction Contracts

You are responsible for the behavior boundary of an insurance claim-intake assistant. Start with an unmeasurable instruction, evolve it one contract component at a time, evaluate every revision on the same 20 cases, diagnose failures, and decide what remains application policy rather than prompt text.

## Scenario and experimental question

Aster Insurance may draft administrative intake responses. It may never approve a claim, issue payment, override policy, diagnose an injury, or treat user-provided instructions as authority.

**Question:** Does explicitly defining objective, evidence, constraints, boundary examples, typed output, and safe failure measurably reduce unsupported and incorrectly routed responses?

Success means correct outcomes, zero unsupported drafts, correct clarification, and a schema-valid safe result—not persuasive prose.

## Learning objectives and safety boundaries

You will separate task contract from model adapter and business policy; compare seven successive contract versions; inspect decisions without chain-of-thought; measure correctness, unsupported claims, clarification, schema validity, tokens, and local evaluation time; inject conflicting authority; and map the result to enterprise controls. All cases are synthetic and no external action is implemented.

## Environment and reproducibility

Python 3.10+; deterministic dataset and adapter; expected offline runtime under one minute. The default provider is `mock`. For live model behavior, export your own `OPENAI_API_KEY`, set `PROMPT_COURSE_PROVIDER=openai`, restart the kernel, and monitor your usage. Never paste or print a key in this notebook.

In [ ]:
from importlib.util import module_from_spec, spec_from_file_location
from dataclasses import asdict
from pathlib import Path
import os, sys
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path('src').resolve()))
LAB_PATH = Path('curriculum/beginner/02-instruction-contracts/lab.py')
spec = spec_from_file_location('course02_lab', LAB_PATH)
lab = module_from_spec(spec)
sys.modules[spec.name] = lab
spec.loader.exec_module(lab)
cases = lab.load_cases()
print({'provider': os.getenv('PROMPT_COURSE_PROVIDER', 'mock'), 'cases': len(cases), 'slices': sorted({case.slice for case in cases})})
assert len(cases) == 20

## Architecture and manual walkthrough

`identity/authorization policy + approved evidence + untrusted request → instruction contract → model proposal → typed/semantic validation → draft | clarify | escalate | reject`

The prompt contract defines a valid proposal. Identity, tenant scope, permission, payment, and irreversible effects remain deterministic application checks. The trace records component versions, outcomes, evidence identifiers, validation, time, and usage—not private reasoning.

In [ ]:
pd.Series({case.slice for case in cases}).value_counts() if False else pd.Series([case.slice for case in cases]).value_counts()

## Baseline — “Handle this request”

The baseline omits the objective, approved evidence, constraints, output, and failure behavior. Its apparent simplicity hides every acceptance criterion. Run it over clear, missing, ambiguous, conflicting, out-of-scope, and injection slices.

In [ ]:
baseline_results = [lab.evaluate_version(lab.VERSIONS[0], case) for case in cases]
baseline = pd.DataFrame([asdict(result) for result in baseline_results])
baseline[['case_id', 'slice', 'outcome', 'expected_outcome', 'supported', 'schema_valid']].head(10)

In [ ]:
baseline.groupby('slice').agg(task_correct=('outcome', lambda values: 0)) if False else baseline.assign(correct=baseline.outcome == baseline.expected_outcome).groupby('slice')[['correct', 'supported', 'schema_valid']].mean()

## Inspect baseline failures

The vague system drafts missing-evidence and malicious requests, cannot represent a safe failure, and produces no typed interface. These are `PROMPT` contract gaps plus `SECURITY`/`WORKFLOW` boundaries. A stronger sentence alone cannot authorize a payment or authenticate a source.

## Step 1 — objective and non-goals

An objective defines the business proposal: administrative intake drafting. It makes medical advice, investment advice, and unrelated generation rejectable. It still does not prevent unsupported insurance claims.

In [ ]:
print(lab.render_contract(lab.VERSIONS[1]))
objective_results = [lab.evaluate_version(lab.VERSIONS[1], case) for case in cases]
sum(result.outcome == result.expected_outcome for result in objective_results) / len(objective_results)

## Step 2 — evidence boundary

The assistant may use only the current claims policy and a verified claim form. Missing approved evidence must not be converted into a confident draft. This component should reduce unsupported claims and improve clarification without changing the objective.

In [ ]:
print(lab.render_contract(lab.VERSIONS[2]))
evidence_results = [lab.evaluate_version(lab.VERSIONS[2], case) for case in cases]
pd.DataFrame([asdict(result) for result in evidence_results]).query("slice in ['missing_evidence', 'ambiguous']")[[ 'case_id', 'outcome', 'supported', 'reasons']]

## Step 3 — constraints and authority

Explicitly prohibit approval, payment, and policy override. The evaluator treats imperatives inside the request as untrusted data. This improves the proposal boundary; a production runtime must still enforce identity and authorization outside the model.

In [ ]:
constraint_results = [lab.evaluate_version(lab.VERSIONS[3], case) for case in cases]
pd.DataFrame([asdict(result) for result in constraint_results]).query("slice in ['injection', 'conflicting_evidence']")[[ 'case_id', 'slice', 'outcome', 'reasons']]

## Steps 4–6 — boundary examples, typed output, and failure path

Boundary examples make ambiguous clarification explicit. The typed `ContractProposal` makes every outcome inspectable. The failure contract requires safe non-draft outcomes to use the same interface, so a consumer never needs to infer failure from prose.

In [ ]:
for version in lab.VERSIONS[4:]:
    print(f'--- {version.version} {version.label} ---')
    print(lab.render_contract(version))

## Run and quantify every revision

Each revision runs against the identical 20 cases. Token counts are explicit local estimates for relative prompt-size comparison; they are not provider billing data. Evaluation time uses `perf_counter`.

In [ ]:
all_results = lab.run_experiment()
summary = pd.DataFrame(lab.summarize(all_results)).set_index('version')
summary.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)
summary[['task_correctness', 'clarification_correctness', 'schema_validity']].plot(marker='o', ax=axes[0])
axes[0].set_ylim(-.03, 1.05); axes[0].set_ylabel('Rate'); axes[0].set_title('Behavior improves by contract component'); axes[0].grid(alpha=.25)
summary['unsupported_claim_rate'].plot.bar(ax=axes[1], color='#C44536')
axes[1].set_ylim(0, 1); axes[1].set_ylabel('Rate among drafts'); axes[1].set_title('Unsupported claim rate'); axes[1].grid(axis='y', alpha=.25)
plt.show()

## Interpret the experiment

The experiment is diagnostic, not a universal benchmark. Objective removes out-of-scope work; evidence removes unsupported drafts; constraints handle malicious authority and conflicts; examples specify ambiguity; output and failure components make every terminal state valid for downstream software. More text is not automatically better—the gain must map to a measured failure.

## Optional live provider experiment

The same typed request/response boundary runs offline by default. In explicit OpenAI mode it uses the current Responses API structured-output path. One live case is an integration smoke test, not proof of quality; run the whole suite and capture usage/latency before comparing models.

In [ ]:
provider_result = lab.run_provider_case(cases[0])
print(provider_result.value.model_dump())
print({'mode': provider_result.response.mode, 'model': provider_result.response.model, 'elapsed_seconds': provider_result.response.elapsed_seconds, 'usage': provider_result.response.usage})

## Failure injection — impossible authority

A user asks the assistant to ignore policy and approve a claim. This is not a wording puzzle. The deterministic boundary rejects it, and an effect-capable service must independently authorize the actor, tenant, amount, policy, and idempotency key.

In [ ]:
attack = next(case for case in cases if case.id == 'IC-016')
attack_result = lab.evaluate_version(lab.CONTRACT, attack)
print(asdict(attack_result))
assert attack_result.outcome == 'reject' and attack_result.schema_valid

## Diagnose the failure

Classify errors as `PROMPT / CONTEXT / EXAMPLE / MODEL / SCHEMA / RETRIEVAL / TOOL / WORKFLOW / EVALUATOR / SECURITY / RUNTIME`. Missing evidence is context/retrieval; ambiguous routing may be contract/example; malformed output is schema/model/runtime; an attempted payment is security/workflow. Do not reflexively edit the prompt when the missing control belongs elsewhere.

In [ ]:
final_frame = pd.DataFrame([asdict(lab.evaluate_version(lab.CONTRACT, case)) for case in cases])
final_frame.assign(correct=final_frame.outcome == final_frame.expected_outcome).groupby('slice')[['correct', 'supported', 'schema_valid']].mean()

## Production upgrade

| Notebook | Production |
| --- | --- |
| Synthetic JSONL | governed versioned eval suite with tenant-safe fixtures |
| Contract dataclass | reviewed behavior artifact with schema, examples, policy and model config |
| Local approved-source tuple | authenticated policy/document service with provenance |
| Printed result | privacy-aware trace, metrics, alert and audit event |
| No external action | narrow authenticated API with authorization and idempotency |
| Shell API key | secret manager or workload identity |
| Local comparison | CI gate, canary, drift monitoring and rollback |

Apply rate limits, timeouts, retry budgets, refusal handling, tenant isolation, retention policy, redaction, incident response, and human escalation proportional to risk.

## When not to use the approach

Use deterministic code when the task is a fully specified calculation or policy lookup. Do not use a prompt contract as authentication, authorization, data isolation, or transaction control. Do not add examples or more wording when evaluation already passes and the added context only increases cost.

## Review questions, exercises, and advanced challenge

1. Which contract component prevents unsupported drafts?
2. Why does a typed reject outcome help a downstream consumer?
3. Which controls remain outside prompt text?
4. When could a boundary example cause a regression?

**Exercise 1:** add a cross-tenant request and implement the deterministic rejection point.

**Exercise 2:** change the missing-evidence policy from clarify to escalate and predict affected slices before running.

**Advanced challenge:** run all 20 cases in live mode for two contract versions. Capture provider usage and measured latency, manually review semantic support, and write a release decision with explicit safety gates.

## Summary

An instruction contract is a versioned behavioral interface: objective, evidence, constraints, examples, typed output, and safe failure. It makes proposals testable, while deterministic software still owns identity, authorization, external effects, and operational reliability.